# 시드 반복 — 0단계 vs 1단계(He), 양쪽 프레임워크

지금까지의 결과는 전부 **시드 0 한 번씩**이다. 이 노트북은 같은 조건을 시드 0·1·2로 세 번씩 돌려서 두 가지를 확인한다.

1. **같은 조건에서 결과가 얼마나 흔들리는가** — 이 폭보다 작은 차이는 효과인지 우연인지 가를 수 없다 (튜닝 전제조건 ④)
2. **He 초기화의 효과가 시드와 상관없이 나오는가** — 시드 0에서는 "학습은 빨라지는데 일반화는 그대로"였다

| | 값 |
|---|---|
| 프레임워크 | Keras, PyTorch |
| 초기화 | 0단계 = 프레임워크 기본값(`default`), 1단계 = He(`he`) |
| 시드 | 0, 1, 2 |
| 실행 횟수 | 2 × 2 × 3 = **12번** (시드 0은 이전 결과와 같은지 확인도 겸한다) |
| 그 밖의 조건 | 이전 단계와 같다: 2 Conv(32·64) + Dropout 0.5 + Dense(11), Adam lr 1e-3, batch 32, 5 epoch, 증강 없음 |
| 소요 시간 | 대략 45분 (Keras 한 번에 약 2.5분, PyTorch 한 번에 약 4.5분) |

**시드마다 달라지는 것**: 가중치 초기값, 학습 데이터를 섞는 순서, Dropout이 끄는 뉴런. 모델 구조와 데이터는 그대로다.

## 실행 방법

위에서부터 셀을 차례로 돌린다 (셀 0 → 1 → 2 → 3 → 4).

- 실행이 하나 끝날 때마다 결과가 Drive의 `food11_v2_seeds.csv`에 한 줄씩 쌓인다. **런타임이 끊겨도 앞선 결과는 남는다.** 셀 0·1부터 다시 돌리면 끝난 실행은 건너뛰고 이어서 한다.
- 처음부터 다시 하려면 Drive에서 그 CSV 파일을 지운다.
- 셀 2(Keras)와 셀 3(PyTorch)은 서로 독립이다. 셀 4(요약)는 있는 결과만으로 만든다.
- 실행 중에는 **실행 하나가 끝날 때만** 한 줄이 출력된다. 첫 줄은 Keras 약 3분, PyTorch 약 5분 뒤에 나오고, 그 사이에는 조용한 게 정상이다.
- **셀 4의 출력을 통째로 붙여 주면 된다.**

## 어떻게 읽을까

**흔들림 폭** = 같은 조건(프레임워크·초기화)에서 시드만 바꿨을 때 결과의 최소~최대 범위. 이 폭보다 작은 차이는 효과인지 우연인지 가를 수 없다.

He 초기화에 효과가 있다고 말하려면 둘 다 맞아야 한다.

1. 시드 3개 모두에서 같은 방향으로 움직인다 (셀 4 출력 2번의 "+인 시드"가 3/3 또는 0/3)
2. 평균 차이가 흔들림 폭(출력 2번의 "시드 범위")보다 크다

셀 4는 출력 4개와 그래프 1개를 만든다: ① 조건별 평균 [최소~최대] ② He 효과와 시드 범위 ③ 두 프레임워크 격차 ④ 시드 0 재현 확인 ⑤ 그래프(`seed_bands.png`, 선 = 평균, 띠 = 최소~최대)

### 예상 (가설 — 결과로 확인한다)

1. 같은 조건의 test acc 흔들림은 1~3%p쯤일 것이다. v1에서 같은 조건을 다시 돌렸을 때 최대 4.6%p였는데, 그때는 시드가 고정돼 있지 않았다.
2. He의 **train 쪽 효과**(학습 속도 ↑, 과적합 ↑)는 크기 때문에 모든 시드에서 같은 방향으로 나올 것이다.
3. **val/test acc의 He 전후 차이는 흔들림 폭 안에 들어갈 것이다.** 그러면 "일반화 정확도는 그대로"가 확인된다. 차이가 폭보다 크게 나오면 시드 하나로 본 결론이 틀렸던 것이다.
4. 두 프레임워크의 격차는 기본값보다 He에서 작을 것이다 (시드 0에서는 5에폭 train acc 격차가 23.2%p → 11.0%p).
5. 시드 0을 다시 돌리면 이전과 거의 같은 값이 나올 것이다. GPU 연산 때문에 아주 작은 차이는 있을 수 있다.

## 셀 0 — 데이터 복사 (최초 1회, 런타임 끊기면 다시)

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")

# Drive에서 직접 읽으면 학습이 수십 배 느려져서 로컬로 복사한다
if not os.path.isdir("/content/food11"):
    !cp -r /content/drive/MyDrive/food11 /content/food11

## 셀 1 — 공통 설정과 기록 함수

In [ ]:
import csv
import gc
import json
import os
import time

import numpy as np

DATA_DIR = "/content/food11"
IMG, BATCH, EPOCHS = 128, 32, 5
NUM_CLASSES = 11
N_PARAMS = 653_003           # 두 프레임워크 모두 이 값이어야 같은 모델이다

SEEDS = (0, 1, 2)
INITS = ("default", "he")    # default = 0단계(프레임워크 기본값) · he = 1단계(He 초기화)

# 실행 하나가 끝날 때마다 한 줄씩 쌓는다. 런타임이 끊겨도 남고,
# 다시 돌리면 끝난 실행은 건너뛴다. 처음부터 다시 하려면 이 파일을 지운다.
RESULTS = "/content/drive/MyDrive/food11_v2_seeds.csv"
FIELDS = ["framework", "init", "seed", "n_params", "minutes",
          "train_acc", "train_loss", "val_acc", "val_loss",    # 에폭별 값 (JSON 리스트)
          "test_acc", "test_loss"]


def done_runs():
    if not os.path.exists(RESULTS):
        return set()
    with open(RESULTS, newline="", encoding="utf-8") as f:
        return {(r["framework"], r["init"], int(r["seed"])) for r in csv.DictReader(f)}


def save_row(row):
    is_new = not os.path.exists(RESULTS)
    with open(RESULTS, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if is_new:
            w.writeheader()
        w.writerow(row)


def make_row(framework, init, seed, n_params, minutes, hist, test_loss, test_acc):
    return dict(framework=framework, init=init, seed=seed, n_params=n_params,
                minutes=round(minutes, 2),
                train_acc=json.dumps(hist["train_acc"]),
                train_loss=json.dumps(hist["train_loss"]),
                val_acc=json.dumps(hist["val_acc"]),
                val_loss=json.dumps(hist["val_loss"]),
                test_acc=test_acc, test_loss=test_loss)


def report(row):
    tr = json.loads(row["train_acc"])[-1]
    va = json.loads(row["val_acc"])[-1]
    print(f"[{row['framework']} · {row['init']} · 시드 {row['seed']}] "
          f"train {tr:.3f} · val {va:.3f} · test {float(row['test_acc']):.3f} "
          f"· {row['minutes']:.1f}분")

## 셀 2 — Keras 6번 (시드 3 × 초기화 2) · 약 16분

In [ ]:
import keras
from keras import layers


def run_keras(init, seed):
    keras.utils.set_random_seed(seed)
    init_kw = dict(kernel_initializer="he_normal") if init == "he" else {}

    def load(split, shuffle):
        ds = keras.utils.image_dataset_from_directory(
            f"{DATA_DIR}/{split}", image_size=(IMG, IMG),
            batch_size=BATCH, shuffle=shuffle, seed=seed)
        rescale = layers.Rescaling(1.0 / 255)
        return ds.map(lambda x, y: (rescale(x), y))

    train_ds = load("training", True)
    val_ds = load("validation", False)
    test_ds = load("evaluation", False)

    # 128 -> conv 126 -> pool 63 -> conv 61 -> pool 30
    model = keras.Sequential([
        keras.Input(shape=(IMG, IMG, 3)),

        layers.Conv2D(32, 3, activation="relu", **init_kw),
        layers.MaxPooling2D(2),

        layers.Conv2D(64, 3, activation="relu", **init_kw),
        layers.MaxPooling2D(2),

        layers.Flatten(),
        layers.Dropout(0.5),
        layers.Dense(NUM_CLASSES, activation="softmax", **init_kw),
    ])
    n_params = int(sum(np.prod(w.shape) for w in model.trainable_weights))
    assert n_params == N_PARAMS, n_params

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    t0 = time.time()
    h = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=0).history
    test_loss, test_acc = model.evaluate(test_ds, verbose=0)
    hist = {"train_acc": h["accuracy"], "train_loss": h["loss"],
            "val_acc": h["val_accuracy"], "val_loss": h["val_loss"]}
    row = make_row("keras", init, seed, n_params, (time.time() - t0) / 60,
                   {k: [float(v) for v in vs] for k, vs in hist.items()},
                   float(test_loss), float(test_acc))
    keras.backend.clear_session()
    gc.collect()
    return row


for seed in SEEDS:
    for init in INITS:
        if ("keras", init, seed) in done_runs():
            print(f"[keras · {init} · 시드 {seed}] 이미 있음 — 건너뜀")
            continue
        row = run_keras(init, seed)
        save_row(row)
        report(row)

## 셀 3 — PyTorch 6번 (시드 3 × 초기화 2) · 약 27분

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

tf_ = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.ToTensor(),          # 0~1 변환 + HWC -> CHW
])


def run_torch(init, seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    g = torch.Generator().manual_seed(seed)

    def load(split, shuffle):
        ds = datasets.ImageFolder(f"{DATA_DIR}/{split}", transform=tf_)
        return DataLoader(ds, batch_size=BATCH, shuffle=shuffle,
                          generator=g if shuffle else None,
                          num_workers=2, pin_memory=True)

    train_dl = load("training", True)
    val_dl = load("validation", False)
    test_dl = load("evaluation", False)

    # 128 -> conv 126 -> pool 63 -> conv 61 -> pool 30
    model = nn.Sequential(
        nn.Conv2d(3, 32, 3), nn.ReLU(), nn.MaxPool2d(2, 2),
        nn.Conv2d(32, 64, 3), nn.ReLU(), nn.MaxPool2d(2, 2),
        nn.Flatten(),
        nn.Dropout(0.5),
        nn.Linear(64 * 30 * 30, NUM_CLASSES),
    ).to(device)

    if init == "he":      # 1단계 — 가중치만. bias는 PyTorch 기본값 그대로
        for m in model:
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight, mode="fan_in", nonlinearity="relu")

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    assert n_params == N_PARAMS, n_params

    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    def evaluate(dataloader):
        model.eval()
        total, correct, n = 0.0, 0, 0
        with torch.no_grad():
            for X, y in dataloader:
                X, y = X.to(device), y.to(device)
                pred = model(X)
                total += loss_fn(pred, y).item() * y.size(0)
                correct += (pred.argmax(1) == y).sum().item()
                n += y.size(0)
        return total / n, correct / n

    hist = {"train_acc": [], "train_loss": [], "val_acc": [], "val_loss": []}
    t0 = time.time()
    for epoch in range(EPOCHS):
        model.train()
        total, correct, n = 0.0, 0, 0
        for X, y in train_dl:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            loss = loss_fn(pred, y)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            total += loss.item() * y.size(0)
            correct += (pred.argmax(1) == y).sum().item()
            n += y.size(0)
        val_loss, val_acc = evaluate(val_dl)
        hist["train_acc"].append(correct / n)
        hist["train_loss"].append(total / n)
        hist["val_acc"].append(val_acc)
        hist["val_loss"].append(val_loss)

    test_loss, test_acc = evaluate(test_dl)
    row = make_row("torch", init, seed, n_params, (time.time() - t0) / 60,
                   hist, test_loss, test_acc)
    del model, optimizer, train_dl, val_dl, test_dl
    torch.cuda.empty_cache()
    gc.collect()
    return row


for seed in SEEDS:
    for init in INITS:
        if ("torch", init, seed) in done_runs():
            print(f"[torch · {init} · 시드 {seed}] 이미 있음 — 건너뜀")
            continue
        row = run_torch(init, seed)
        save_row(row)
        report(row)

## 셀 4 — 요약 (이 셀의 출력을 붙여 주면 된다)

In [ ]:
import unicodedata

import matplotlib.pyplot as plt

with open(RESULTS, newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
assert rows, "결과가 아직 없다 — 셀 2·3을 먼저 돌릴 것"
assert all(int(r["n_params"]) == N_PARAMS for r in rows), "파라미터 수가 다른 실행이 있다"

FW = {"keras": "Keras", "torch": "PyTorch"}
INIT_NAME = {"default": "기본값", "he": "He"}


def width(s):
    return sum(2 if unicodedata.east_asian_width(c) in "WF" else 1 for c in s)


def pad(s, w):
    return s + " " * max(0, w - width(s))


def metrics(r):
    ta, tl = json.loads(r["train_acc"]), json.loads(r["train_loss"])
    va, vl = json.loads(r["val_acc"]), json.loads(r["val_loss"])
    return dict(train_acc=ta[-1] * 100, train_loss=tl[-1],
                val_acc=va[-1] * 100, val_loss=vl[-1],
                best_val_loss=min(vl), best_ep=int(np.argmin(vl)) + 1,
                test_acc=float(r["test_acc"]) * 100, test_loss=float(r["test_loss"]))


runs = {}      # (프레임워크, 초기화) -> {시드: 지표}
for r in rows:
    runs.setdefault((r["framework"], r["init"]), {})[int(r["seed"])] = metrics(r)


def spread(group, key):
    v = [m[key] for m in group.values()]
    return max(v) - min(v)


def bar(title):
    print()
    print("=" * 100)
    print(title)
    print("=" * 100)


# ---- 1. 조건별 결과 ---------------------------------------------------------
KEYS = [(fw, i) for fw in FW for i in INITS if (fw, i) in runs]
ROWS = [("train_acc", "5에폭 train acc (%)", 1), ("train_loss", "5에폭 train loss", 2),
        ("val_acc", "5에폭 val acc (%)", 1), ("val_loss", "5에폭 val loss", 2),
        ("best_val_loss", "val loss 최저", 2), ("best_ep", "  최저였던 에폭", 1),
        ("test_acc", "test acc (%)", 1), ("test_loss", "test loss", 2)]


def span(vals, nd):
    v = np.array(vals, dtype=float)
    return f"{v.mean():.{nd}f} [{v.min():.{nd}f}~{v.max():.{nd}f}]"


bar("1. 조건별 결과 — 평균 [최소~최대]")
print(pad("", 22) + "".join(
    pad(f"{FW[fw]} {INIT_NAME[i]} (n={len(runs[(fw, i)])})", 21) for fw, i in KEYS))
for key, label, nd in ROWS:
    print(pad(label, 22) + "".join(
        pad(span([m[key] for m in runs[k].values()], nd), 21) for k in KEYS))

# ---- 2. He 초기화의 효과 ----------------------------------------------------
EFFECT = [("train_acc", "5에폭 train acc", "%p"), ("val_acc", "5에폭 val acc", "%p"),
          ("test_acc", "test acc", "%p"), ("train_loss", "5에폭 train loss", ""),
          ("val_loss", "5에폭 val loss", ""), ("test_loss", "test loss", "")]

bar("2. He 초기화의 효과 (He − 기본값, 같은 시드끼리 짝지어 뺀 값)\n"
    "   효과로 보려면: 모든 시드에서 방향이 같고 ('+인 시드'가 n/n 또는 0/n),\n"
    "   평균 차이의 크기가 시드 범위보다 크다")
shown = False
for fw in FW:
    d, h = runs.get((fw, "default")), runs.get((fw, "he"))
    if not d or not h:
        continue
    shown = True
    seeds = sorted(set(d) & set(h))
    print(f"{FW[fw]}  (짝지은 시드 {seeds})")
    print("  " + pad("지표", 18) + pad("평균 차이", 12) + pad("최소~최대", 18)
          + pad("+인 시드", 10) + "시드 범위 (기본값 / He)")
    for key, label, unit in EFFECT:
        diff = np.array([h[s][key] - d[s][key] for s in seeds])
        print("  " + pad(label, 18)
              + pad(f"{diff.mean():+.2f}{unit}", 12)
              + pad(f"{diff.min():+.2f}~{diff.max():+.2f}", 18)
              + pad(f"{int((diff > 0).sum())}/{len(diff)}", 10)
              + f"{spread(d, key):.2f} / {spread(h, key):.2f}")
if not shown:
    print("(기본값과 He 결과가 모두 있어야 계산된다)")

# ---- 3. 두 프레임워크 격차 --------------------------------------------------
bar("3. 두 프레임워크 격차 (Keras − PyTorch, 같은 시드끼리)")
shown = False
for i in INITS:
    k, t = runs.get(("keras", i)), runs.get(("torch", i))
    if not k or not t:
        continue
    shown = True
    seeds = sorted(set(k) & set(t))
    for key, label in (("train_acc", "5에폭 train acc"),
                       ("train_loss", "5에폭 train loss"), ("test_acc", "test acc")):
        gap = np.array([k[s][key] - t[s][key] for s in seeds])
        unit = "%p" if "acc" in key else ""
        print(pad(f"{INIT_NAME[i]}  {label}", 28)
              + f"{gap.mean():+.2f}{unit} [{gap.min():+.2f}~{gap.max():+.2f}]")
if not shown:
    print("(Keras와 PyTorch 결과가 모두 있어야 계산된다)")

# ---- 4. 시드 0 재현 확인 ----------------------------------------------------
PREV = {   # 이전 노트북에서 시드 0으로 한 번 돌렸을 때 (소수 넷째 자리까지 찍었던 값)
    ("keras", "default"): dict(train_acc=0.7160, test_acc=0.4694, test_loss=1.7778),
    ("keras", "he"):      dict(train_acc=0.7729, test_acc=0.4625, test_loss=1.8589),
    ("torch", "default"): dict(train_acc=0.4839, test_acc=0.4156, test_loss=1.7235),
    ("torch", "he"):      dict(train_acc=0.6631, test_acc=0.4290, test_loss=1.8580),
}
bar("4. 시드 0 재현 확인 — 같은 시드로 다시 돌렸을 때 이전 결과와 같은가")
for r in rows:
    k = (r["framework"], r["init"])
    if int(r["seed"]) != 0 or k not in PREV:
        continue
    now = dict(train_acc=json.loads(r["train_acc"])[-1],
               test_acc=float(r["test_acc"]), test_loss=float(r["test_loss"]))
    parts = []
    for key, prev in PREV[k].items():
        if round(now[key], 4) == prev:
            parts.append(f"{key} 같음")
        else:
            parts.append(f"{key} {now[key]:.4f} (이전 {prev:.4f})")
    print(pad(f"{FW[k[0]]} {INIT_NAME[k[1]]}", 16) + " · ".join(parts))

# ---- 5. 그래프 --------------------------------------------------------------
ACCENT, DARKGY, GRID, MUTED = "#16785C", "#3A4353", "#DDD9D2", "#6B7482"
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False
PANELS = [("train_acc", "train acc"), ("val_acc", "val acc"), ("val_loss", "val loss")]
fw_list = [fw for fw in FW if any(k[0] == fw for k in KEYS)]
ep = np.arange(1, EPOCHS + 1)

fig, axes = plt.subplots(len(fw_list), 3, figsize=(13, 3.5 * len(fw_list)),
                         dpi=150, squeeze=False)
for i, fw in enumerate(fw_list):
    for j, (key, name) in enumerate(PANELS):
        ax = axes[i][j]
        for init, color, ls, lab in (("default", DARKGY, "--", "default"),
                                     ("he", ACCENT, "-", "He")):
            rs = [r for r in rows if r["framework"] == fw and r["init"] == init]
            if not rs:
                continue
            curves = np.array([json.loads(r[key]) for r in rs])      # (시드, 에폭)
            ax.fill_between(ep, curves.min(0), curves.max(0), color=color,
                            alpha=0.18, lw=0)
            ax.plot(ep, curves.mean(0), color=color, ls=ls, lw=2.0, marker="o",
                    ms=3.5, label=f"{lab} (n={len(rs)})")
        ax.set_title(f"{FW[fw]} · {name}", fontsize=11, color=DARKGY)
        ax.set_xlabel("Epoch", fontsize=9.5, color=MUTED)
        ax.set_xticks(ep)
        ax.grid(True, color=GRID, lw=0.8, alpha=0.9)
        ax.set_axisbelow(True)
        for s in ("top", "right"):
            ax.spines[s].set_visible(False)
        for s in ("left", "bottom"):
            ax.spines[s].set_color(GRID)
        ax.tick_params(colors=MUTED, labelsize=9)
        ax.legend(frameon=False, fontsize=9)

fig.suptitle("default init vs He init  -  line = mean, band = min~max over seeds",
             fontsize=13, color=DARKGY)
fig.tight_layout(rect=(0, 0, 1, 0.95))
fig.savefig("seed_bands.png", facecolor="white")
plt.show()
print("[그래프] seed_bands.png")